<a href="https://colab.research.google.com/github/Nyamu007/Completed-Projects/blob/main/rob21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pybullet numpy matplotlib imageio imageio-ffmpeg

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Simulation Environment & Robot Setup

In [2]:
import pybullet as p
import pybullet_data
import numpy as np
import time
import imageio
from IPython.display import HTML
from base64 import b64encode

# 1. Connect to PyBullet in DIRECT (headless) mode for Colab
if p.isConnected():
    p.disconnect()
physicsClient = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

# 2. Load Ground Plane & Table
planeId = p.loadURDF("plane.urdf")
tableId = p.loadURDF("table/table.urdf", basePosition=[0.5, 0, 0], globalScaling=0.8)
TABLE_HEIGHT = 0.50  # Top surface height of scaled table

# 3. Load a Standard Robot Arm (Kuka IIWA or Franka Panda)
# We use Franka Panda because it includes a working two-finger gripper out of the box
robotStartPos = [0.1, 0, TABLE_HEIGHT]
robotId = p.loadURDF("franka_panda/panda.urdf", basePosition=robotStartPos, useFixedBase=True)

# Panda joint indices: 0-6 are arm joints, 9 and 10 are gripper fingers
ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
GRIPPER_JOINTS = [9, 10]
END_EFFECTOR_INDEX = 11

# Reset arm to a neutral "ready" pose
neutral_pose = [0, -0.4, 0, -2.2, 0, 1.8, 0.8]
for i, joint_idx in enumerate(ARM_JOINTS):
    p.resetJointState(robotId, joint_idx, neutral_pose[i])

# 4. Spawn Different Items to Pick
items = {}

# Item 1: Red Cube
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.025, 0.025, 0.025])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.025, 0.025, 0.025], rgbaColor=[0.9, 0.1, 0.1, 1])
items["Red Cube"] = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=cube_col, baseVisualShapeIndex=cube_vis,
                                      basePosition=[0.55, -0.2, TABLE_HEIGHT + 0.03])

# Item 2: Green Cylinder
cyl_col = p.createCollisionShape(p.GEOM_CYLINDER, radius=0.025, height=0.06)
cyl_vis = p.createVisualShape(p.GEOM_CYLINDER, radius=0.025, length=0.06, rgbaColor=[0.1, 0.8, 0.2, 1])
items["Green Cylinder"] = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=cyl_col, baseVisualShapeIndex=cyl_vis,
                                            basePosition=[0.65, 0.0, TABLE_HEIGHT + 0.035])

# Item 3: Blue Sphere (High friction so it doesn't roll away easily)
sph_col = p.createCollisionShape(p.GEOM_SPHERE, radius=0.025)
sph_vis = p.createVisualShape(p.GEOM_SPHERE, radius=0.025, rgbaColor=[0.1, 0.3, 0.9, 1])
items["Blue Sphere"] = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=sph_col, baseVisualShapeIndex=sph_vis,
                                         basePosition=[0.55, 0.2, TABLE_HEIGHT + 0.03])
p.changeDynamics(items["Blue Sphere"], -1, rollingFriction=0.05, lateralFriction=1.0)

# Visual Marker for Drop-off Zone
drop_zone_pos = [0.35, 0.35, TABLE_HEIGHT + 0.005]
zone_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.08, 0.08, 0.002], rgbaColor=[1.0, 0.8, 0.0, 0.6])
p.createMultiBody(baseVisualShapeIndex=zone_vis, basePosition=drop_zone_pos)

print("Environment initialized with items:", list(items.keys()))

Environment initialized with items: ['Red Cube', 'Green Cylinder', 'Blue Sphere']


Inverse Kinematics, Gripper Control & Video Recorder

In [4]:
frames = []

def capture_frame():
    """Renders a camera view from the side-front angle."""
    view_matrix = p.computeViewMatrixFromYawPitchRoll(
        cameraTargetPosition=[0.45, 0.05, TABLE_HEIGHT + 0.1],
        distance=1.1,
        yaw=45,
        pitch=-30,
        roll=0,
        upAxisIndex=2
    )
    proj_matrix = p.computeProjectionMatrixFOV(fov=60, aspect=640/480, nearVal=0.1, farVal=100.0)
    _, _, rgb, _, _ = p.getCameraImage(width=640, height=480, viewMatrix=view_matrix, projectionMatrix=proj_matrix)
    frames.append(np.reshape(rgb, (480, 640, 4))[:, :, :3])

def step_simulation(steps=60, record_every=8):
    """Advances physics and records frames."""
    for i in range(steps):
        p.stepSimulation()
        if i % record_every == 0:
            capture_frame()

def control_gripper(open_gripper=True):
    """Opens (0.04m) or closes (0.0m) the parallel gripper."""
    target_pos = 0.04 if open_gripper else 0.0
    force = 20 if open_gripper else 60  # Higher force when grasping
    for joint in GRIPPER_JOINTS:
        p.setJointMotorControl2(robotId, joint, p.POSITION_CONTROL, targetPosition=target_pos, force=force)
    step_simulation(steps=80)

def move_to(target_pos, steps=120):
    """Uses Inverse Kinematics (IK) to move the end-effector pointing straight down."""
    # Orientation pointing straight down (-Z axis)
    down_orientation = p.getQuaternionFromEuler([np.pi, 0, 0])
    joint_poses = p.calculateInverseKinematics(robotId, END_EFFECTOR_INDEX, target_pos, down_orientation)

    for i, joint_idx in enumerate(ARM_JOINTS):
        p.setJointMotorControl2(robotId, joint_idx, p.POSITION_CONTROL, targetPosition=joint_poses[i], force=200)
    step_simulation(steps=steps)

def pick_and_place(item_name, item_id, drop_pos):
    """Full pick-and-place routine for a given object."""
    pos, _ = p.getBasePositionAndOrientation(item_id)
    print(f"Picking {item_name} at {np.round(pos, 3)}...")

    # 1. Open gripper & hover above item
    control_gripper(open_gripper=True)
    move_to([pos[0], pos[1], pos[2] + 0.15])

    # 2. Descend to grasp height
    move_to([pos[0], pos[1], pos[2] + 0.01])

    # 3. Close gripper
    control_gripper(open_gripper=False)

    # 4. Lift item
    move_to([pos[0], pos[1], pos[2] + 0.20])

    # 5. Move above drop zone
    move_to([drop_pos[0], drop_pos[1], drop_pos[2] + 0.20])

    # 6. Lower and release
    move_to([drop_pos[0], drop_pos[1], drop_pos[2] + 0.05])
    control_gripper(open_gripper=True)

    # 7. Retract upward
    move_to([drop_pos[0], drop_pos[1], drop_pos[2] + 0.20])

Pick-and-Place Loop & Render Video

In [5]:
# Define slight offsets in the drop zone for each item
drop_offsets = [
    [0.32, 0.32, TABLE_HEIGHT],
    [0.38, 0.35, TABLE_HEIGHT],
    [0.32, 0.38, TABLE_HEIGHT]
]

# Execute pick-and-place for each item
for idx, (name, obj_id) in enumerate(items.items()):
    pick_and_place(name, obj_id, drop_offsets[idx])

# Save frames as MP4
video_path = "robot_pick_and_place.mp4"
imageio.mimsave(video_path, frames, fps=30)

# Display video inline in Colab
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=640 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>')

Picking Red Cube at [ 0.55 -0.2   0.53]...
Picking Green Cylinder at [ 0.65 -0.    0.53]...
Picking Blue Sphere at [0.55  0.2   0.525]...
